# 04 — Three zones instead of one threshold

Instead of a single "flagged at 0.4" cut-off, every posting lands in one of three zones:

| Zone | Rule for its cut-off (chosen on the **training** set) |
|---|---|
| 🔴 likely scam | lowest cut-off where **≥ 90%** of postings at or above it are scams |
| 🟢 likely real | highest cut-off where the zone lets through **≤ 5% of all scams** |
| 🟡 unsure | everything in between |

The model itself is **not retrained**; this notebook only chooses where to draw the lines.
All choices use the training set; the test set is **not** used here (see section 6).

In [1]:
import os

# Limit each worker to one math thread. Without this, parallel cross-validation
# oversubscribed the CPU in notebook 03 and slowed to a crawl.
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1"

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict

DATA_PATH = Path("..") / "data" / "fake_job_postings.csv"
MODEL_PATH = Path("..") / "models" / "job_trust_v1.joblib"
RANDOM_STATE = 42

SCAM_ZONE_MIN_PRECISION = 0.90    # "likely scam" must be right at least 9 times in 10
REAL_ZONE_MAX_SHARE_OF_SCAMS = 0.05  # "likely real" may let through at most 5% of all scams

## 1. Re-create the same data and split as notebooks 02 and 03

In [2]:
df = pd.read_csv(DATA_PATH)

TEXT_COLUMNS = ["title", "company_profile", "description", "requirements", "benefits"]
df["text"] = df[TEXT_COLUMNS].fillna("").agg(" ".join, axis=1)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
df["salary_listed"] = df["salary_range"].notna().astype(int)
df["has_company_profile"] = df["company_profile"].notna().astype(int)

bundle = joblib.load(MODEL_PATH)
METADATA_COLUMNS = bundle["metadata_columns"]

X = df[["text"] + METADATA_COLUMNS]
y = df["fraudulent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(y_train):,} ({y_train.mean():.2%} fake) | Test: {len(y_test):,} ({y_test.mean():.2%} fake)")
# 🔒 X_test / y_test are not used anywhere below.

Train: 12,852 (4.47% fake) | Test: 3,214 (4.48% fake)


## 2. Cross-validated probabilities on the training set

`clone` makes an **untrained copy** of the final pipeline with exactly the same settings
(C=10, words + two-word phrases, metadata). Each training posting is then scored by a copy
that never saw it, the same fair method notebook 03 used to choose 0.4.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

train_probs = cross_val_predict(
    clone(bundle["pipeline"]), X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]
print(f"Scored {len(train_probs):,} training postings")

Scored 12,852 training postings


## 3. Find the two cut-offs

We try every cut-off from 0.01 to 0.99 in steps of 0.01:
- **Scam cut-off**: the *lowest* one where postings at or above it are ≥ 90% scams.
- **Real cut-off**: the *highest* one where postings below it hold ≤ 5% of all training scams.

In [4]:
y_arr = y_train.to_numpy()
total_scams = y_arr.sum()

rows = []
for t in np.round(np.arange(0.01, 1.00, 0.01), 2):
    above, below = train_probs >= t, train_probs < t
    rows.append({
        "cutoff": t,
        "precision above": y_arr[above].mean() if above.any() else np.nan,
        "share of scams below": y_arr[below].sum() / total_scams,
        "scam rate below": y_arr[below].mean() if below.any() else np.nan,
    })
sweep = pd.DataFrame(rows)

LIKELY_SCAM_AT = sweep.loc[sweep["precision above"] >= SCAM_ZONE_MIN_PRECISION, "cutoff"].min()
LIKELY_REAL_BELOW = sweep.loc[sweep["share of scams below"] <= REAL_ZONE_MAX_SHARE_OF_SCAMS, "cutoff"].max()
assert LIKELY_REAL_BELOW < LIKELY_SCAM_AT, "zones overlap"

print(f"likely scam : probability >= {LIKELY_SCAM_AT}")
print(f"unsure      : {LIKELY_REAL_BELOW} <= probability < {LIKELY_SCAM_AT}")
print(f"likely real : probability < {LIKELY_REAL_BELOW}")

likely scam : probability >= 0.79
unsure      : 0.12 <= probability < 0.79
likely real : probability < 0.12


**What this tells us:** "Likely scam" starts at 0.79, which is where flags become right at least 9 times in 10, and "likely real" ends at 0.12, so anything between 0.12 and 0.79 is "unsure." Both lines are far from the old single 0.4 cut-off: we now only shout "scam" when the model's track record is strong, and only reassure when the score is very low.

## 4. What lands in each zone (training set, cross-validated)

In [5]:
def zone_table(probs, labels, real_below, scam_at) -> pd.DataFrame:
    """Per zone: how many postings, how many scams, scam share, and share of ALL scams."""
    probs = np.asarray(probs)
    zones = np.where(probs >= scam_at, "likely scam", np.where(probs < real_below, "likely real", "unsure"))
    frame = pd.DataFrame({"zone": zones, "scam": np.asarray(labels)})
    table = frame.groupby("zone")["scam"].agg(postings="size", scams="sum")
    table = table.reindex(["likely real", "unsure", "likely scam"]).fillna(0).astype(int)
    table["% of postings"] = (table["postings"] / table["postings"].sum() * 100).round(1)
    table["scam share in zone"] = (table["scams"] / table["postings"] * 100).round(2)
    table["% of all scams"] = (table["scams"] / table["scams"].sum() * 100).round(1)
    return table


zone_table(train_probs, y_train, LIKELY_REAL_BELOW, LIKELY_SCAM_AT)

,postings,scams,% of postings,scam share in zone,% of all scams
zone,,,,,
likely real,11455,28,89.1,0.24,4.9
unsure,954,144,7.4,15.09,25.1
likely scam,443,402,3.4,90.74,70.0


**What this tells us:** Green covers 89% of postings with only 0.24% scams (about 1 in 400) and lets through 4.9% of all scams; red is 90.7% scams and catches 70% of them; the remaining 25% of scams land in yellow. So 95% of scams get either a warning or a "check carefully," at the cost of 7.4% of postings showing as unsure, which is the trade-off we want for students who can't easily judge postings themselves.

## 5. Why not "≤ 1% of the green zone is scams"?

That was the first rule we tried. It sounds safe, but the green zone is so large that the
many real postings dilute it: the rule is met even when green swallows a big share of scams.

In [6]:
first_rule_cutoff = sweep.loc[sweep["scam rate below"] <= 0.01, "cutoff"].max()
print(f"First rule's green cut-off: below {first_rule_cutoff} (higher than the old 0.4 flag!)")
zone_table(train_probs, y_train, first_rule_cutoff, LIKELY_SCAM_AT)

First rule's green cut-off: below 0.62 (higher than the old 0.4 flag!)


,postings,scams,% of postings,scam share in zone,% of all scams
zone,,,,,
likely real,12292,122,95.6,0.99,21.3
unsure,117,50,0.9,42.74,8.7
likely scam,443,402,3.4,90.74,70.0


**What this tells us:** The first rule's green zone was only 1% scams on paper, but it stretched up to 0.62 and quietly labeled 21% of all scams "likely real," worse than the old 0.4 flag. The lesson: a zone's scam *rate* can look tiny just because the zone is huge; what matters to a student is how many scams the zone lets through.

## 6. Test set: not used here

The test set was scored in notebook 03 (its one planned use). The **first** rule above was
then checked on it once, before we saw that rule was flawed. That check gave: "likely scam"
90.3% scams (113 postings) and "likely real" holding 22.2% of all test scams, matching the
training numbers in section 5.

The current rule was chosen from training numbers only and is deliberately **not** checked on
the test set: a third look would make the test numbers mean even less. The README lists this
as a limitation.